In [8]:
import requests
from requests.adapters import HTTPAdapter, Retry
import json
import re

# Session with automatic retry on server errors (500, 502, 503, 504)
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

In [9]:
def get_next_link(headers):
    if "Link" in headers:
        re_next_link = re.compile(r'<(.+)>; rel="next"')
        match = re_next_link.match(headers["Link"])
        if match:
            return match.group(1)

def get_batch(batch_url):
    while batch_url:
        response = session.get(batch_url)
        response.raise_for_status()
        # Get total number of entries from response header
        total = response.headers.get("x-total-results", 0)
        yield response, total
        batch_url = get_next_link(response.headers)

In [10]:
# Positive dataset URL
url_positive = "https://rest.uniprot.org/uniprotkb/search?compressed=false&format=json&query=%28%28reviewed%3Atrue%29+AND+%28fragment%3Afalse%29+AND+%28taxonomy_id%3A2759%29+AND+%28existence%3A1%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28ft_signal_exp%3A*%29%29&size=500"

In [11]:
def filter_entry_positive(entry):
    """
    Returns True if the entry passes all filters for the positive dataset.
    Checks:
    - Has a Signal feature
    - Evidence is experimental (ECO:0000269)
    - Cleavage site is known (not null, not UNKNOWN)
    - Description is not "Not cleaved"
    - SP length >= 14 (end position >= 14)
    """
    for feature in entry["features"]:
        # Only look at Signal peptide features
        if feature["type"] == "Signal":

            # Check description — exclude "Not cleaved"
            description = feature.get("description", "")
            if "Not cleaved" in description:
                return False

            # Check end position — must not be null or UNKNOWN
            end = feature["location"]["end"]["value"]
            if end is None:
                return False
            if feature["location"]["end"]["modifier"] == "UNKNOWN":
                return False

            # Check SP length >= 14
            if end < 14:
                return False

            # Check experimental evidence (ECO:0000269)
            evidences = feature.get("evidences", [])
            for evidence in evidences:
                if evidence["evidenceCode"] == "ECO:0000269":
                    return True  # passes all checks

    return False  # no valid signal peptide found

In [12]:
def get_kingdom(entry):
    """
    Determines the eukaryotic kingdom from the organism lineage.
    Returns one of: Metazoa, Fungi, Viridiplantae, Other
    """
    lineage = entry.get("organism", {}).get("lineage", [])
    if "Metazoa" in lineage:
        return "Metazoa"
    elif "Fungi" in lineage:
        return "Fungi"
    elif "Viridiplantae" in lineage:
        return "Viridiplantae"
    else:
        return "Other"


def extract_fields_positive(entry):
    """
    Extracts the fields we need for the positive TSV:
    - accession
    - organism name
    - kingdom
    - protein length
    - SP cleavage position (end of signal peptide)
    """
    accession = entry["primaryAccession"]
    organism  = entry["organism"]["scientificName"]
    kingdom   = get_kingdom(entry)
    length    = entry["sequence"]["length"]

    # Get the SP cleavage position from features
    cleavage_pos = None
    for feature in entry["features"]:
        if feature["type"] == "Signal":
            cleavage_pos = feature["location"]["end"]["value"]
            break  # take the first one

    return (accession, organism, kingdom, length, cleavage_pos)

In [13]:
def get_dataset_positive(search_url, filter_function, extract_function,
                          output_tsv, output_fasta):
    """
    Fetches the positive dataset from UniProt and saves it as TSV and FASTA.
    """
    n_total, n_filtered = 0, 0

    with open(output_tsv, "w") as tsv_out, open(output_fasta, "w") as fasta_out:

        # Write TSV header
        print("accession", "organism", "kingdom", "length", "cleavage_pos",
              sep="\t", file=tsv_out)

        # Fetch all batches
        for batch, total in get_batch(search_url):
            batch_json = json.loads(batch.text)

            for entry in batch_json["results"]:
                n_total += 1

                # Apply filter
                if filter_function(entry):
                    n_filtered += 1

                    # Extract TSV fields
                    fields = extract_function(entry)
                    print(*fields, sep="\t", file=tsv_out)

                    # Write FASTA
                    accession = entry["primaryAccession"]
                    sequence  = entry["sequence"]["value"]
                    print(f">{accession}", file=fasta_out)
                    print(sequence, file=fasta_out)

    print(f"Done! Total retrieved: {n_total}, passed filter: {n_filtered}")
    print(f"TSV saved to: {output_tsv}")
    print(f"FASTA saved to: {output_fasta}")

In [14]:
# Output file names
output_tsv_positive   = "positive_dataset.tsv"
output_fasta_positive = "positive_dataset.fasta"

# Run!
get_dataset_positive(
    url_positive,
    filter_entry_positive,
    extract_fields_positive,
    output_tsv_positive,
    output_fasta_positive
)

Done! Total retrieved: 2972, passed filter: 2957
TSV saved to: positive_dataset.tsv
FASTA saved to: positive_dataset.fasta


The Nagitve dataset

In [15]:
url_negative = "https://rest.uniprot.org/uniprotkb/search?compressed=false&format=json&query=%28%28reviewed%3Atrue%29+AND+%28fragment%3Afalse%29+AND+%28taxonomy_id%3A2759%29+AND+%28existence%3A1%29+AND+%28length%3A%5B40+TO+*%5D%29+NOT+%28ft_signal%3A*%29+AND+%28cc_scl_term_exp%3ASL-0091+OR+cc_scl_term_exp%3ASL-0173+OR+cc_scl_term_exp%3ASL-0191+OR+cc_scl_term_exp%3ASL-0209+OR+cc_scl_term_exp%3ASL-0204+OR+cc_scl_term_exp%3ASL-0039%29%29&size=500"

In [16]:
def filter_entry_negative(entry):
    """
    Returns True if the entry passes all filters for the negative dataset.
    Checks:
    - No signal peptide feature at any evidence level
    """
    for feature in entry.get("features", []):
        if feature["type"] == "Signal":
            return False  # has an SP → reject
    return True  # no SP found → keep

In [17]:
def has_tm_in_first_90(entry):
    """
    Checks if a transmembrane helix starts within the first 90 residues.
    Returns True or False.
    """
    for feature in entry.get("features", []):
        if feature["type"] == "Transmembrane":
            start = feature["location"]["start"]["value"]
            if start is not None and start <= 90:
                return True
    return False


def extract_fields_negative(entry):
    """
    Extracts the fields we need for the negative TSV:
    - accession
    - organism name
    - kingdom
    - protein length
    - TM helix in first 90 residues (True/False)
    """
    accession = entry["primaryAccession"]
    organism  = entry["organism"]["scientificName"]
    kingdom   = get_kingdom(entry)
    length    = entry["sequence"]["length"]
    has_tm    = has_tm_in_first_90(entry)

    return (accession, organism, kingdom, length, has_tm)

In [18]:
def get_dataset_negative(search_url, filter_function, extract_function,
                          output_tsv, output_fasta):
    """
    Fetches the negative dataset from UniProt and saves it as TSV and FASTA.
    """
    n_total, n_filtered = 0, 0

    with open(output_tsv, "w") as tsv_out, open(output_fasta, "w") as fasta_out:

        # Write TSV header
        print("accession", "organism", "kingdom", "length", "has_tm_in_90",
              sep="\t", file=tsv_out)

        for batch, total in get_batch(search_url):
            batch_json = json.loads(batch.text)

            for entry in batch_json["results"]:
                n_total += 1

                if filter_function(entry):
                    n_filtered += 1

                    fields = extract_function(entry)
                    print(*fields, sep="\t", file=tsv_out)

                    accession = entry["primaryAccession"]
                    sequence  = entry["sequence"]["value"]
                    print(f">{accession}", file=fasta_out)
                    print(sequence, file=fasta_out)

    print(f"Done! Total retrieved: {n_total}, passed filter: {n_filtered}")
    print(f"TSV saved to: {output_tsv}")
    print(f"FASTA saved to: {output_fasta}")

In [19]:
def get_dataset_negative(search_url, filter_function, extract_function,
                          output_tsv, output_fasta):
    """
    Fetches the negative dataset from UniProt and saves it as TSV and FASTA.
    """
    n_total, n_filtered = 0, 0

    with open(output_tsv, "w") as tsv_out, open(output_fasta, "w") as fasta_out:

        # Write TSV header
        print("accession", "organism", "kingdom", "length", "has_tm_in_90",
              sep="\t", file=tsv_out)

        for batch, total in get_batch(search_url):
            batch_json = json.loads(batch.text)

            for entry in batch_json["results"]:
                n_total += 1

                if filter_function(entry):
                    n_filtered += 1

                    fields = extract_function(entry)
                    print(*fields, sep="\t", file=tsv_out)

                    accession = entry["primaryAccession"]
                    sequence  = entry["sequence"]["value"]
                    print(f">{accession}", file=fasta_out)
                    print(sequence, file=fasta_out)

    print(f"Done! Total retrieved: {n_total}, passed filter: {n_filtered}")
    print(f"TSV saved to: {output_tsv}")
    print(f"FASTA saved to: {output_fasta}")

In [20]:
output_tsv_negative   = "negative_dataset.tsv"
output_fasta_negative = "negative_dataset.fasta"

get_dataset_negative(
    url_negative,
    filter_entry_negative,
    extract_fields_negative,
    output_tsv_negative,
    output_fasta_negative
)

Done! Total retrieved: 20975, passed filter: 20975
TSV saved to: negative_dataset.tsv
FASTA saved to: negative_dataset.fasta
